# Sensitivity

Which of an ensemble recipe's uncertain inputs actually move the outcome, and *when* ([ADR-007](../docs/architecture/ADR-007-sensitivity-analysis.md)).

- A **one-at-a-time** batch (`sim run <recipe> --sensitivity`) moves each varying row to a low and a high quantile of its own range, holding the others at their center. That makes the effects directly comparable.
- A **Monte Carlo** batch (`sim run <recipe>`) answers a different question: with everything varying at once, which rows could be pinned to a literal without losing anything?

The terminal equivalent is `sim ensemble sensitivity <recipe>` (plus `--attribute NAME` for the "why" view). See [docs/guides/ensembles.md](../docs/guides/ensembles.md) for how to set up and size each kind of run.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present -- see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.


In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.notebook_ui.sensitivity_widgets import sensitivity_widget
from analysis.sim.notebook_ui.pickers import find_state_dir


In [ ]:
find_state_dir()

## The parameters being varied

The parameters are the recipe's **varying rows**: ordinary `calendar.csv` rows with `vary_distribution` set, each marking one uncertain value. Every `set_rate`, `set_amount`, `one_off_transfer` or `retire_scheduled` row can vary either its **date** or its **amount** (`vary_target`), drawn from:

- `uniform`, between `vary_low` and `vary_high`: any value in the range is equally plausible. This suits dates.
- `truncated_normal`, around `vary_mean` with spread `vary_sd`, clipped to `vary_low`/`vary_high` (±3 SD when they're left blank): values near the center are more likely.

Add one with, for example:

```sh
sim calendar add set-rate 2027-01-31 brokerage_growth 6.0 --scenario my_recipe --name growth \
  --vary-target amount --vary-distribution truncated_normal --vary-mean 6.0 --vary-sd 2.0
```

Each row's **center** (a uniform range's midpoint, a truncated normal's mean) is the base case. A one-at-a-time batch holds every row there while it moves one row to the low and high quantiles of its own range, so the ranges you choose decide what "plausibly low" and "plausibly high" mean. A careless, too-wide range will make a row look more important than it is.

The **Parameters** table at the top of the widget below lists the selected batch's rows: what each acts on, its distribution, range and center, and the values the batch actually used. It's read from the batch's own files, so it describes what was run even if the recipe has changed since. See [ADR-006 §2](../docs/architecture/ADR-006-ensemble-scenarios.md) for every `vary_*` column.


## Explore a batch

- **Recipe / Batch**: any recipe with a generated batch; the newest batch is selected. Re-run the cell below after generating a new one.
- **Dollar basis / Delta**: nominal or today's (CPI-constant) dollars, and dollars or % of the base. They apply to every chart below.

- **Parameters**: the selected batch's varying rows and the values it used (see above).

For a **one-at-a-time** batch:

- **Overview**: parameters x years. *Share* shows which parameter dominates when; *signed swing* shows how big and in which direction.
- **At one year**: each parameter's change at its low and high quantile, with the numbers below.
- **Over time**: one parameter's change against the base, one line per quantile. A fund running out, a retirement date or compounding growth each have a recognizable shape.
- **Why**: the same change broken down exactly, by income/expense category, by individual account, or by where the money ends up.
- **Across the range** (only with 4 or more quantiles, `--quantiles`): the response across the parameter's range at one year. A step or a sharp bend is a threshold inside the range.

For a **Monte Carlo** batch: R² by year (how far per-parameter numbers can be trusted), then each row's rank correlation with net worth and its 95% interval, and which rows are pin candidates.


In [ ]:
sensitivity_widget()